# 7. Leak-Free Downstream Evaluation

**Pipeline per fold (no data leakage):**
1. Split → train / test (from `Split_seed_0..9`)
2. Batch correction: fit on **train only** → transform both
3. PCA: fit on **train only** → transform both
4. ML model: train on train → predict on test → metrics

**Models:** LogReg + LGBM (with inner GridSearchCV) | TabPFN (optional)

**Pipelines compared:**
- COMPASS_PT (raw) — no correction
- COMPASS_PT + Adv2-cVAE — adversarial batch correction
- _(Future: Geneformer, Harmony, etc.)_

**Baseline:** DummyClassifier (majority class)


## 0. Environment Setup

In [ ]:
# !rm -rf batchcor-rna-embeds/
!git clone -b feat/scvi-batch-correction https://github.com/onion-42/batchcor-rna-embeds.git 2>/dev/null || (cd batchcor-rna-embeds && git pull origin feat/scvi-batch-correction)
%cd batchcor-rna-embeds

!pip install -q uv
!uv pip install --system -e .
!pip install -q lightgbm
# !pip install -q tabpfn  # uncomment if using TabPFN


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from loguru import logger

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import (
    BATCH_COL, SEED,
    COMPASS_PT_EMBEDDING_KEY, SCVI_LATENT_DIM,
)
from batchcor_rna_emb.data_io import load_cohort
from batchcor_rna_emb.modeling.fold_runner import FoldConfig, run_experiment
from batchcor_rna_emb.modeling.plotting import (
    plot_pipeline_comparison, plot_all_results,
)

set_logger(level='INFO')
logger.info('Imports OK')


## 1. Configuration

In [ ]:
DATA_INTERIM_DIR = Path('/content/drive/MyDrive/data/interim')
RESULTS_DIR = Path('/content/drive/MyDrive/data/results')
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

COHORTS = [
    'KIRC_Tissue_ICI_Pred',
        'PUB_BRCA_SCANB',
]

TARGETS = ['Response']  # Add 'Target_OS_bin', 'Target_PFS_bin' later
MODELS = ['logreg', 'lgbm']  # Add 'tabpfn' if installed
N_SPLITS = 10
CVAE_EPOCHS = 100

# --- Define all pipelines to compare ---
# Each entry: (embedding_key, embedding_label, correction_method, correction_label)
PIPELINES = [
    (COMPASS_PT_EMBEDDING_KEY, 'COMPASS_PT', 'none', ''),
    (COMPASS_PT_EMBEDDING_KEY, 'COMPASS_PT', 'cvae_adv2', 'Adv2-cVAE'),
    # Future: add Geneformer, Harmony, etc.
    # ('FM_Geneformer_embedding', 'Geneformer', 'none', ''),
    # ('FM_Geneformer_embedding', 'Geneformer', 'harmony', 'Harmony'),
]

logger.info('Cohorts: {}', COHORTS)
logger.info('Pipelines: {}', [(p[1], p[3] or 'raw') for p in PIPELINES])


## 2. Run Experiments
For each cohort × target × pipeline, run all folds × all models.
Transformations (PCA, cVAE) are cached per fold and shared across models.

In [ ]:
all_dfs = []

for cohort_name in COHORTS:
    zarr_path = DATA_INTERIM_DIR / f'{cohort_name}.adata.zarr'
    
    logger.info('\n' + '=' * 70)
    logger.info('Loading cohort: {}', cohort_name)
    
    try:
        adata = load_cohort(zarr_path)
    except Exception as e:
        logger.warning('Failed to load {}: {}', cohort_name, e)
        continue
    
    logger.info('Loaded: {} samples', adata.n_obs)
    
    for target_col in TARGETS:
        if target_col not in adata.obs.columns:
            logger.warning("Target '{}' not in {}, skipping", target_col, cohort_name)
            continue
        
        for emb_key, emb_label, corr_method, corr_label in PIPELINES:
            if emb_key not in adata.obsm:
                logger.warning("Embedding '{}' not in {}, skipping", emb_key, cohort_name)
                continue
            
            cfg = FoldConfig(
                embedding_key=emb_key,
                embedding_label=emb_label,
                target_col=target_col,
                correction_method=corr_method,
                correction_label=corr_label,
                n_pca=128,
                cvae_epochs=CVAE_EPOCHS,
                seed=SEED,
            )
            
            logger.info('\n--- {} | {} | {} ---', cohort_name, target_col, cfg.pipeline_label)
            
            df = run_experiment(
                adata, cfg,
                model_names=MODELS,
                n_splits=N_SPLITS,
            )
            df['cohort'] = cohort_name
            all_dfs.append(df)

if all_dfs:
    results = pd.concat(all_dfs, ignore_index=True)
    logger.info('\nTotal results: {} rows', len(results))
    display(results.head(20))
else:
    logger.error('No results generated!')
    results = pd.DataFrame()


## 3. Save Raw Tables
Save ALL per-fold results so you can regenerate plots without re-running experiments.

In [ ]:
if not results.empty:
    results.to_csv(RESULTS_DIR / 'leak_free_per_fold_results.csv', index=False)
    logger.info('Saved per-fold results to {}', RESULTS_DIR / 'leak_free_per_fold_results.csv')
    
    # Summary table: mean ± std
    metric_cols = ['roc_auc', 'pr_auc', 'f1', 'f1_weighted', 'f1_macro', 'balanced_accuracy']
    group_cols = ['cohort', 'target', 'pipeline', 'model']
    
    existing_metrics = [c for c in metric_cols if c in results.columns]
    summary = results.groupby(group_cols)[existing_metrics].agg(['mean', 'std']).round(4)
    summary.columns = [f'{m}_{s}' for m, s in summary.columns]
    summary = summary.reset_index()
    
    summary.to_csv(RESULTS_DIR / 'leak_free_summary.csv', index=False)
    logger.info('Saved summary to {}', RESULTS_DIR / 'leak_free_summary.csv')
    display(summary)


## 4. Main Figures: F1 weighted
One plot per (Model, Cohort). Bars sorted by mean, swarm overlay,
DummyClassifier baseline, Wilcoxon+Holm brackets between neighbors.

In [ ]:
if not results.empty:
    figs = plot_all_results(
        results,
        metric_col='f1_weighted',
        metric_label='F1 weighted',
        pipeline_col='pipeline',
        split_col='split',
        save_dir=FIGURES_DIR / 'main',
    )
    for fig in figs:
        plt.show()


## 5. Supplement: ROC-AUC
Same format as main figures, but with ROC-AUC.

In [ ]:
if not results.empty:
    figs_roc = plot_all_results(
        results,
        metric_col='roc_auc',
        metric_label='ROC-AUC',
        pipeline_col='pipeline',
        split_col='split',
        save_dir=FIGURES_DIR / 'supplement_roc_auc',
    )
    for fig in figs_roc:
        plt.show()


## 6. Supplement: Cross-Validation Scores
Inner CV best ROC-AUC for LogReg/LGBM (hyperparameter selection quality).

In [ ]:
if not results.empty and 'cv_best_roc_auc' in results.columns:
    cv_results = results.dropna(subset=['cv_best_roc_auc']).copy()
    
    if not cv_results.empty:
        figs_cv = plot_all_results(
            cv_results,
            metric_col='cv_best_roc_auc',
            metric_label='Inner CV ROC-AUC (best)',
            pipeline_col='pipeline',
            split_col='split',
            save_dir=FIGURES_DIR / 'supplement_cv',
        )
        for fig in figs_cv:
            plt.show()
    else:
        logger.warning('No CV scores available')


## 7. Quick Reload (optional)
If you already ran the experiment and just want to re-plot from saved CSV.

In [ ]:
# Uncomment to reload saved results and re-plot without re-running experiments:
# results = pd.read_csv(RESULTS_DIR / 'leak_free_per_fold_results.csv')
# figs = plot_all_results(results, metric_col='f1_weighted', metric_label='F1 weighted',
#                         save_dir=FIGURES_DIR / 'main')
# for fig in figs: plt.show()
